# 04 — EDA dataset gabungan (`merged.parquet`)

Memeriksa hasil merge tiga sumber (blibli, tokopedia, tokopedia2025) yang dibuat
di Drive, **sebelum** dipakai melatih apa pun.

Yang diperiksa, berurutan:

| # | Bagian | Pertanyaan yang dijawab |
|---|---|---|
| 1 | Muat & bentuk | jumlah baris/kolom cocok dengan `MERGED_DATASET.md`? |
| 2 | Kelengkapan | mana yang `null`, mana yang string kosong, mana yang list kosong — per sumber |
| 3 | Identitas | `product_id` unik? awalan sumber konsisten? id sintetis berapa? |
| 4 | Duplikat | `dup_url`/`dup_judul` cocok kalau dihitung ulang? |
| 5 | Harga | null, 0, dan pencilan per kategori (MAD pada log-harga) |
| 6 | Deskripsi | panjang prosa, `""` vs `null`, mojibake, sisa HTML |
| 7 | Kategori | sebaran `kategori_umkm`, audit `kategori_asal` |
| 8 | Gambar | path bisa diresolusi? berkasnya ada? bisa dibuka? ukurannya wajar? |
| 9 | Contoh gambar | grid visual, cek mata sendiri |
| 10 | Vonis | daftar temuan PASS/WARN/FAIL + simpan JSON |

Notebook ini **read-only** terhadap dataset. Satu-satunya tulisan adalah
`data/merged/eda_merged_report.json`.

## 1. Setup & muat

`ROOT_MAP` memetakan akar Colab (`/content/drive/MyDrive/IAC/...`) ke folder lokal.
Sumber yang tidak dipetakan **tidak dianggap rusak** — dilaporkan terpisah sebagai
"tidak dipetakan", karena gambarnya memang ada di Drive orang lain.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MERGED = PROJECT / "data" / "merged" / "merged.parquet"
REPORT = PROJECT / "data" / "merged" / "eda_merged_report.json"

# akar Colab -> akar lokal. Tambah entri kalau kamu sudah menyalin gambar temanmu.
ROOT_MAP = {
    "/content/drive/MyDrive/IAC/tokopedia_dataset": PROJECT / "data",
    # "/content/drive/MyDrive/IAC/blibli": Path("D:/IAC/blibli"),
    # "/content/drive/MyDrive/IAC/data/external/tokopedia2025": Path("D:/IAC/tokopedia2025"),
}

SAMPLE_IMAGES = 300   # berkas per sumber yang benar-benar dibuka dengan PIL

# klaim dari MERGED_DATASET.md, untuk dicocokkan bukan dipercaya
CLAIM_ROWS = 28_443
CLAIM_PER_SOURCE = {"blibli": 8_800, "tokopedia": 18_443, "tokopedia2025": 1_200}

assert MERGED.exists(), f"tidak ada {MERGED} — unduh dulu dari folder Drive 'merged'"
df = pd.read_parquet(MERGED)
print(f"{MERGED.name}: {len(df):,} baris x {df.shape[1]} kolom, "
      f"{df.memory_usage(deep=True).sum() / 1e6:.0f} MB di memori")

In [ ]:
FINDINGS = []

def catat(level, bagian, pesan):
    """Kumpulkan temuan. level: PASS | WARN | FAIL."""
    FINDINGS.append({"level": level, "bagian": bagian, "pesan": pesan})
    print(f"[{level:4}] {pesan}")

def is_list(x):
    return isinstance(x, (list, tuple, np.ndarray))

def list_len(x):
    return len(x) if is_list(x) else 0

LIST_COLS = ["category_path", "image_urls", "local_image_paths"]
TEXT_COLS = ["title", "description", "url", "location", "brand",
             "merchant_name", "search_keyword", "kategori_umkm", "kategori_asal"]

# Yang benar-benar dipakai model: judul, harga, deskripsi, gambar, kategori.
# Kolom di luar ini (rating, sold_count, toko, dll) hanya bonus — kekosongannya
# dicatat sebagai info, tidak pernah jadi FAIL/WARN.
CORE_COLS = ["title", "price", "description",
             "image_urls", "local_image_paths",
             "category_path", "kategori_umkm"]

print(df.dtypes.to_string())

In [ ]:
# bentuk vs klaim dokumentasi
if len(df) == CLAIM_ROWS:
    catat("PASS", "bentuk", f"jumlah baris {len(df):,} sama dengan klaim MERGED_DATASET.md")
else:
    catat("FAIL", "bentuk", f"baris {len(df):,} != klaim {CLAIM_ROWS:,}")

vc = df["source"].value_counts()
for src_, klaim in CLAIM_PER_SOURCE.items():
    nyata = int(vc.get(src_, 0))
    if nyata == klaim:
        catat("PASS", "bentuk", f"{src_}: {nyata:,} baris, sesuai klaim")
    else:
        catat("FAIL", "bentuk", f"{src_}: {nyata:,} baris, klaim {klaim:,}")

tak_diklaim = set(vc.index) - set(CLAIM_PER_SOURCE)
if tak_diklaim:
    catat("WARN", "bentuk", f"sumber tak terdokumentasi: {sorted(tak_diklaim)}")

# Folder Drive IAC juga berisi shopee/ dan tokopedia_listings/, tapi keduanya tidak
# ikut merge. Alasannya sah: ekspornya tidak punya field inti.
#   shopee            -> product_id, image, name, shop_name, kategori. Tanpa harga, tanpa deskripsi.
#   tokopedia_listings-> Nama Produk, Harga, Rating, Terjual, URL. Tanpa deskripsi, tanpa gambar.
tak_ikut = {"shopee", "tokopedia_listings"} - set(vc.index)
if tak_ikut:
    catat("PASS", "lingkup",
          f"sumber {sorted(tak_ikut)} ada di Drive tapi sengaja tidak digabung — "
          "ekspornya tanpa deskripsi/gambar, tidak berguna untuk auto-description")

vc.to_frame("baris").assign(persen=lambda d: (100 * d.baris / len(df)).round(1))

## 2. Kelengkapan — tiga jenis "kosong"

`null` (tidak tersedia di ekspor sumber), `""` (penjual menulisnya sebagai gambar),
dan `[]` (list kosong) adalah hal berbeda. Dicek terpisah, per sumber.

In [ ]:
def profil_kosong(g):
    out = {}
    for c in g.columns:
        if c in LIST_COLS:
            out[c] = float(100 * (g[c].map(list_len) == 0).mean())
        elif c in TEXT_COLS:
            s = g[c].astype("object")
            kosong = s.isna() | (s.fillna("x").astype(str).str.strip() == "")
            out[c] = float(100 * kosong.mean())
        else:
            out[c] = float(100 * g[c].isna().mean())
    return pd.Series(out)

kosong = pd.DataFrame({src: profil_kosong(g) for src, g in df.groupby("source")}).round(1)
kosong["TOTAL"] = profil_kosong(df).round(1)
print("persen kosong (null / string kosong / list kosong) per kolom per sumber\n")
display(kosong.sort_values("TOTAL", ascending=False))

In [ ]:
# kolom inti wajib terisi untuk semua sumber
INTI = ["product_id", "title", "source", "kategori_umkm"]
for c in INTI:
    if kosong.loc[c, "TOTAL"] == 0:
        catat("PASS", "kelengkapan", f"{c}: 0% kosong")
    else:
        catat("FAIL", "kelengkapan", f"{c}: {kosong.loc[c, 'TOTAL']}% kosong — kolom inti")

# kolom inti lain yang kosong total di satu sumber
for c in CORE_COLS:
    if c in INTI or c not in kosong.index:
        continue
    per_src = kosong.loc[c].drop("TOTAL")
    penuh_kosong = per_src[per_src >= 99.9].index.tolist()
    if penuh_kosong:
        catat("WARN", "kelengkapan",
              f"{c}: kosong total di {penuh_kosong} — kolom inti, saring per sumber")

# kolom di luar lingkup (rating, sold, toko, dll): info saja
luar = [c for c in kosong.index if c not in CORE_COLS and c not in INTI]
info = kosong.loc[luar]
info = info[(info.drop(columns="TOTAL") >= 99.9).any(axis=1)]
if len(info):
    print("\ncatatan (di luar lingkup judul/harga/deskripsi/gambar/kategori) — "
          "persen kosong per sumber:")
    display(info)

In [ ]:
# === Banding merged vs ekspor sumber asli (folder Drive IAC) ===
# Hanya field inti: judul, harga, deskripsi, gambar, kategori.
# Kolom lain (rating, sold_count, toko, dll) di luar lingkup, tidak dinilai.
SUMBER_DIR = PROJECT / "data" / "merged" / "sumber"

def _as_list(x):
    if is_list(x):
        return list(x)
    if isinstance(x, str):
        s = x.strip()
        if s.startswith("["):
            try:
                return json.loads(s)
            except Exception:
                return [s]
        return [s] if s else []
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []
    return [x]

def _rangkum(id_, title, price, desc, kat, url, path):
    """Satu bentuk seragam untuk semua sumber."""
    return pd.DataFrame({
        "id": [str(v) for v in id_],
        "title": [("" if v is None or (isinstance(v, float) and np.isnan(v)) else str(v)).strip()
                  for v in title],
        "price": pd.to_numeric(pd.Series(list(price)), errors="coerce").to_numpy(),
        "desc_len": [len("" if v is None or (isinstance(v, float) and np.isnan(v)) else str(v))
                     for v in desc],
        "n_kat": [len(_as_list(v)) for v in kat],
        "n_url": [len(_as_list(v)) for v in url],
        "n_path": [len(_as_list(v)) for v in path],
    }).set_index("id")

def muat_sumber():
    src = {}

    p = PROJECT / "data" / "exports" / "products_slim_ready.jsonl"
    if p.exists():
        d = pd.read_json(p, lines=True)
        src["tokopedia"] = _rangkum(d["product_id"], d["title"], d["price"], d["description"],
                                    d["category_path"], d["image_urls"], d["local_image_paths"])

    p = SUMBER_DIR / "blibli_products.parquet"
    if p.exists():
        d = pd.read_parquet(p)
        src["blibli"] = _rangkum(d["product_id"], d["title"], d["price"], d["description"],
                                 d["category_path"], d["image_urls"], d["local_image_paths"])

    p = SUMBER_DIR / "tokopedia2025_products.csv"
    if p.exists():
        d = pd.read_csv(p)
        # sumber ini tidak punya URL CDN sama sekali; 'images' berisi path lokal relatif
        src["tokopedia2025"] = _rangkum(d["id"], d["name"], d["price"], d["description"],
                                        d["category_name"], [[]] * len(d), d["images"])
    return src

SUMBER = muat_sumber()
print("ekspor sumber yang ketemu:", {k: len(v) for k, v in SUMBER.items()})

In [ ]:
ringkas_banding = []
for s, ref in SUMBER.items():
    m = df[df["source"] == s].copy()
    m["_id"] = m["product_id_asli"].astype(str)
    ada = m[m["_id"].isin(ref.index)]

    hilang_di_merged = len(ref) - len(ada)
    ekstra_di_merged = len(m) - len(ada)

    a = ref.loc[ada["_id"]]
    judul_beda = int((ada["title"].fillna("").astype(str).str.strip().to_numpy()
                      != a["title"].to_numpy()).sum())
    hm = pd.to_numeric(ada["price"], errors="coerce").astype("float64").to_numpy()
    ha = pd.to_numeric(a["price"], errors="coerce").astype("float64").to_numpy()
    harga_beda = int((~np.isclose(hm, ha, equal_nan=True)).sum())
    desc_susut = int((ada["description"].fillna("").astype(str).str.len().to_numpy()
                      < a["desc_len"].to_numpy()).sum())
    kat_susut = int((ada["category_path"].map(list_len).to_numpy() < a["n_kat"].to_numpy()).sum())
    gbr_m = (ada["image_urls"].map(list_len) + ada["local_image_paths"].map(list_len)).to_numpy()
    gbr_a = (a["n_url"] + a["n_path"]).to_numpy()
    gbr_susut = int((gbr_m < gbr_a).sum())

    ringkas_banding.append({
        "sumber": s, "baris_ekspor": len(ref), "baris_merged": len(m), "id_cocok": len(ada),
        "hilang_di_merged": hilang_di_merged, "tak_ada_di_ekspor": ekstra_di_merged,
        "judul_beda": judul_beda, "harga_beda": harga_beda,
        "deskripsi_menyusut": desc_susut, "kategori_menyusut": kat_susut,
        "gambar_menyusut": gbr_susut,
    })

banding_sumber = pd.DataFrame(ringkas_banding).set_index("sumber")
display(banding_sumber)

for s, r in banding_sumber.iterrows():
    n = max(int(r["id_cocok"]), 1)
    if r["id_cocok"] == r["baris_ekspor"] == r["baris_merged"]:
        catat("PASS", "banding", f"{s}: semua {int(r['baris_ekspor']):,} baris ekspor ada di merged, id cocok semua")
    else:
        catat("WARN", "banding",
              f"{s}: ekspor {int(r['baris_ekspor']):,} vs merged {int(r['baris_merged']):,}, "
              f"{int(r['hilang_di_merged']):,} tidak terbawa, {int(r['tak_ada_di_ekspor']):,} tak ada di ekspor")

    for field in ["judul_beda", "harga_beda", "deskripsi_menyusut", "kategori_menyusut", "gambar_menyusut"]:
        jml = int(r[field])
        pct = 100 * jml / n
        if jml == 0:
            catat("PASS", "banding", f"{s}: {field.replace('_', ' ')} — tidak ada")
        elif pct >= 1:
            catat("FAIL", "banding",
                  f"{s}: {field.replace('_', ' ')} pada {jml:,} baris ({pct:.1f}%) — "
                  "isi field inti berubah/berkurang saat merge")
        else:
            catat("WARN", "banding", f"{s}: {field.replace('_', ' ')} pada {jml:,} baris ({pct:.2f}%)")

## 3. Identitas baris

In [ ]:
n_dup_id = int(df["product_id"].duplicated().sum())
if n_dup_id == 0:
    catat("PASS", "identitas", "product_id unik di seluruh baris")
else:
    catat("FAIL", "identitas", f"{n_dup_id:,} product_id duplikat")
    display(df[df["product_id"].duplicated(keep=False)]
            .sort_values("product_id")[["product_id", "source", "title"]].head(10))

# awalan harus 'source:'
awalan = df["product_id"].astype(str).str.split(":").str[0]
awalan_salah = awalan.to_numpy() != df["source"].astype(str).to_numpy()
if awalan_salah.any():
    catat("FAIL", "identitas", f"{int(awalan_salah.sum()):,} product_id tanpa awalan sumber yang benar")
else:
    catat("PASS", "identitas", "semua product_id berawalan 'source:'")

# id yang sama muncul di dua sumber berbeda -> kandidat produk kembar
lintas = int((df.groupby("product_id_asli")["source"].nunique() > 1).sum())
catat("WARN" if lintas else "PASS", "identitas",
      f"{lintas:,} product_id_asli muncul di lebih dari satu sumber")

print("\nid sintetis (SHA-1 dari URL) per sumber:")
print(df.groupby("source")["id_sintetis"].sum().to_string())

## 4. Duplikat — hitung ulang, jangan percaya flag

Bocornya train/test terjadi lewat judul dan URL yang kembar, bukan lewat `product_id`.

In [ ]:
def norm_judul(s):
    return (s.astype("object").fillna("").astype(str).str.lower()
             .str.replace(r"[^a-z0-9 ]+", " ", regex=True)
             .str.replace(r"\s+", " ", regex=True).str.strip())

def norm_url(s):
    return (s.astype("object").fillna("").astype(str).str.split("?").str[0]
             .str.rstrip("/").str.lower())

j = norm_judul(df["title"])
u = norm_url(df["url"])

hitung_dup_judul = j.duplicated(keep=False) & (j != "")
hitung_dup_url = u.duplicated(keep=False) & (u != "")

banding = pd.DataFrame({
    "flag_tersimpan": [int(df["dup_judul"].sum()), int(df["dup_url"].sum())],
    "hitung_ulang": [int(hitung_dup_judul.sum()), int(hitung_dup_url.sum())],
}, index=["dup_judul", "dup_url"])
banding["selisih"] = banding["hitung_ulang"] - banding["flag_tersimpan"]
display(banding)

for nama, beda in banding["selisih"].items():
    if beda == 0:
        catat("PASS", "duplikat", f"{nama} cocok saat dihitung ulang")
    else:
        catat("WARN", "duplikat",
              f"{nama} beda {beda:+,} dari flag tersimpan — normalisasinya tidak sama, "
              "pakai hitungan sendiri saat split")

print("\nbaris tersisa kalau dedupe pada judul ternormalisasi:",
      f"{int((~j.duplicated()).sum()):,} dari {len(df):,}")
print("judul paling sering diulang:")
display(j[j != ""].value_counts().head(8).to_frame("jumlah"))

## 5. Harga

Pencilan dicari **per `kategori_umkm`** dengan MAD pada log-harga. Rp5 juta wajar
untuk elektronik, tidak wajar untuk sayuran — ambang global akan salah di dua arah.

In [ ]:
harga = pd.to_numeric(df["price"], errors="coerce").astype("Float64")
n_null = int(harga.isna().sum())
n_nol = int((harga == 0).sum())
n_neg = int((harga < 0).sum())

catat("WARN" if n_null else "PASS", "harga", f"{n_null:,} harga null")
catat("WARN" if n_nol else "PASS", "harga", f"{n_nol:,} harga bernilai 0")
catat("FAIL" if n_neg else "PASS", "harga", f"{n_neg:,} harga negatif")

ringkas = (df.assign(_h=harga.astype("float64")).groupby("source")["_h"]
             .agg(baris="size", null=lambda s: int(s.isna().sum()),
                  nol=lambda s: int((s == 0).sum()),
                  min="min", median="median",
                  p99=lambda s: s.quantile(0.99), maks="max"))
display(ringkas)

In [ ]:
sah = harga.astype("float64")
sah = sah.where(sah > 0)
log_h = np.log10(sah)

def skor_mad(x):
    med = x.median()
    mad = (x - med).abs().median()
    if not mad or np.isnan(mad):
        return pd.Series(0.0, index=x.index)
    return 0.6745 * (x - med) / mad

skor = log_h.groupby(df["kategori_umkm"]).transform(skor_mad)
pencilan = skor.abs() > 3.5
catat("WARN", "harga",
      f"{int(pencilan.sum()):,} harga pencilan ({100 * pencilan.mean():.1f}%) "
      "pada |MAD-z| > 3.5 per kategori — saring sebelum latih regresi harga")

display(df.loc[pencilan, ["source", "kategori_umkm", "title", "price"]]
          .sort_values("price", ascending=False).head(10))

kategori = sorted(df["kategori_umkm"].dropna().unique())
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(log_h.dropna(), bins=60, color="#4C78A8")
ax[0].set_title("sebaran log10(harga), harga > 0")
ax[0].set_xlabel("log10 rupiah")
ax[1].boxplot([log_h[df["kategori_umkm"] == k].dropna() for k in kategori], tick_labels=kategori)
ax[1].set_title("log10(harga) per kategori_umkm")
ax[1].tick_params(axis="x", rotation=60)
plt.tight_layout()
plt.show()

## 6. Deskripsi

Ini kolom target untuk model auto-description, jadi pemeriksaannya paling ketat.

In [ ]:
desc = df["description"].astype("object")
is_null = desc.isna()
is_kosong = (~is_null) & (desc.fillna("x").astype(str).str.strip() == "")
panjang = desc.fillna("").astype(str).str.len()
src = df["source"]

tabel = pd.DataFrame({
    "baris": df.groupby("source").size(),
    "null": is_null.groupby(src).sum(),
    "kosong_str": is_kosong.groupby(src).sum(),
    "ada_prosa": (panjang > 0).groupby(src).sum(),
    "median_char": panjang.where(panjang > 0).groupby(src).median(),
    "p95_char": panjang.where(panjang > 0).groupby(src).quantile(0.95),
    "maks_char": panjang.groupby(src).max(),
})
tabel["persen_prosa"] = (100 * tabel["ada_prosa"] / tabel["baris"]).round(1)
display(tabel)

pendek = (panjang > 0) & (panjang < 30)
catat("WARN" if pendek.sum() else "PASS", "deskripsi",
      f"{int(pendek.sum()):,} deskripsi < 30 karakter — terlalu pendek untuk target latih")
catat("PASS" if is_null.sum() == 0 else "WARN", "deskripsi",
      f"{int(is_null.sum()):,} null (tidak tersedia) vs {int(is_kosong.sum()):,} "
      "string kosong (deskripsi ditulis sebagai gambar) — dua hal berbeda")

plt.figure(figsize=(11, 3.5))
plt.hist(panjang[panjang > 0].clip(upper=5000), bins=60, color="#54A24B")
plt.title("panjang deskripsi (karakter, dipangkas di 5000)")
plt.show()

In [ ]:
# higienis teks: mojibake, sisa HTML, kontak penjual
teks = desc.fillna("").astype(str)
# catatan: pandas 3 memakai mesin regex RE2 (Arrow) — escape \uXXXX tidak didukung,
# jadi karakter mojibake ditulis dengan escape byte biasa.
pola = {
    "mojibake": "\xc3.|\xe2€|\xc2\xa0|\xef\xbf\xbd",
    "tag HTML": r"<[a-zA-Z/][^>]{0,40}>",
    "entitas HTML": r"&(?:amp|nbsp|lt|gt|quot|#[0-9]+);",
    "nomor WA/telepon": r"(?:\+62|08)[0-9]{8,12}",
    "URL di dalam deskripsi": r"https?://",
}
hasil = {}
for nama, p in pola.items():
    kena = teks.str.contains(p, regex=True, na=False)
    hasil[nama] = [int(kena.sum()), round(100 * float(kena.mean()), 2)]
hig = pd.DataFrame(hasil, index=["baris", "persen"]).T.sort_values("baris", ascending=False)
display(hig)

for nama, row in hig.iterrows():
    if row["persen"] >= 1:
        catat("WARN", "teks",
              f"{nama}: {int(row['baris']):,} baris ({row['persen']}%) — bersihkan sebelum latih")

judul_norm = norm_judul(df["title"])
n_judul_pendek = int((judul_norm.str.len() < 10).sum())
catat("WARN" if n_judul_pendek else "PASS", "teks",
      f"{n_judul_pendek:,} judul < 10 karakter setelah normalisasi")

## 7. Kategori terpadu

`kategori_umkm` adalah hasil pemetaan otomatis, bukan label manusia. `kategori_asal`
menyimpan dasar keputusannya — itulah yang diaudit di sini.

In [ ]:
display(pd.crosstab(df["kategori_umkm"], df["source"], margins=True, margins_name="TOTAL"))
display(pd.crosstab(df["kategori_asal"], df["kategori_umkm"]))

n_lainnya = int((df["kategori_umkm"] == "lainnya").sum())
catat("WARN" if n_lainnya / len(df) > 0.2 else "PASS", "kategori",
      f"'lainnya' {n_lainnya:,} baris ({100 * n_lainnya / len(df):.1f}%) — sisa yang tidak terpetakan")

lemah = df["kategori_asal"].isin(["kata_kunci_judul", "tidak_terpetakan"])
catat("WARN", "kategori",
      f"{int(lemah.sum()):,} baris ({100 * lemah.mean():.1f}%) dipetakan lewat tebakan kata kunci "
      "judul atau tidak terpetakan — jangan dipakai sebagai label latih tanpa audit manual")

print("\ncontoh acak untuk audit mata sendiri (judul -> kategori, dasar):")
if int(lemah.sum()):
    display(df.loc[lemah, ["title", "kategori_umkm", "kategori_asal", "source"]]
              .sample(min(12, int(lemah.sum())), random_state=0))

## 8. Gambar

Tiga hal berbeda, dipisahkan supaya tidak saling menutupi:

1. **Ada path/URL-nya?** (isi kolom)
2. **Path-nya bisa diresolusi ke mesin ini?** (`ROOT_MAP`)
3. **Berkasnya benar-benar ada, bisa dibuka, ukurannya wajar?** (sampel PIL)

Sumber tanpa entri di `ROOT_MAP` dilaporkan "tidak dipetakan", bukan "hilang".

In [ ]:
n_url = df["image_urls"].map(list_len)
n_path = df["local_image_paths"].map(list_len)

img = pd.DataFrame({
    "baris": df.groupby("source").size(),
    "tanpa_url": (n_url == 0).groupby(src).sum(),
    "tanpa_path": (n_path == 0).groupby(src).sum(),
    "median_url": n_url.groupby(src).median(),
    "median_path": n_path.groupby(src).median(),
    "maks_path": n_path.groupby(src).max(),
    "total_path": n_path.groupby(src).sum(),
})
display(img)

tanpa_gambar = int((n_path == 0).sum())
catat("WARN" if tanpa_gambar else "PASS", "gambar",
      f"{tanpa_gambar:,} baris tanpa satu pun path gambar")

kurang = int(((n_url > 0) & (n_path < n_url)).sum())
catat("WARN" if kurang else "PASS", "gambar",
      f"{kurang:,} baris punya URL lebih banyak daripada berkas lokal — "
      "unduhan gambarnya belum lengkap")

In [ ]:
def remap(p):
    """Path Colab -> path lokal. None kalau akarnya tidak ada di ROOT_MAP."""
    p = str(p).replace("\\", "/")
    for akar, lokal in ROOT_MAP.items():
        akar = akar.rstrip("/")
        if p.startswith(akar + "/"):
            return Path(lokal) / p[len(akar) + 1:]
    return None

baris_src, semua_path = [], []
for s, paths in zip(df["source"], df["local_image_paths"]):
    for p in (paths if is_list(paths) else []):
        baris_src.append(s)
        semua_path.append(str(p))

paths_df = pd.DataFrame({"source": baris_src, "path": semua_path})
paths_df["lokal"] = paths_df["path"].map(remap)
paths_df["dipetakan"] = paths_df["lokal"].notna()
paths_df["ada"] = [bool(p.exists()) if p is not None else False for p in paths_df["lokal"]]

resolusi = paths_df.groupby("source").agg(
    total_path=("path", "size"),
    dipetakan=("dipetakan", "sum"),
    ada_berkas=("ada", "sum"),
)
resolusi["persen_ada_dari_dipetakan"] = (
    100 * resolusi["ada_berkas"] / resolusi["dipetakan"].replace(0, np.nan)).round(1)
display(resolusi)

for s, r in resolusi.iterrows():
    if r["dipetakan"] == 0:
        catat("WARN", "gambar",
              f"{s}: {int(r['total_path']):,} path tidak dipetakan ke mesin ini — "
              "gambarnya ada di Drive orang lain, isi ROOT_MAP kalau mau diperiksa")
    elif r["persen_ada_dari_dipetakan"] >= 99:
        catat("PASS", "gambar", f"{s}: {r['persen_ada_dari_dipetakan']}% berkas gambar ditemukan")
    else:
        catat("FAIL", "gambar",
              f"{s}: hanya {r['persen_ada_dari_dipetakan']}% berkas ditemukan "
              f"({int(r['dipetakan'] - r['ada_berkas']):,} hilang) — path menunjuk ke berkas yang tidak ada")

hilang = paths_df.loc[paths_df["dipetakan"] & ~paths_df["ada"], "path"]
if len(hilang):
    print("\ncontoh path yang dipetakan tapi berkasnya tidak ada:")
    for p in hilang.head(5):
        print(" ", p)

In [ ]:
# buka sampel berkas: rusak? terpotong? terlalu kecil?
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = False

def periksa(p):
    try:
        with Image.open(p) as im:
            im.verify()
        with Image.open(p) as im:
            im.load()
            w, h = im.size
            return {"ok": True, "w": w, "h": h, "mode": im.mode,
                    "format": im.format, "kb": p.stat().st_size / 1024, "error": ""}
    except Exception as e:
        return {"ok": False, "w": 0, "h": 0, "mode": "", "format": "",
                "kb": p.stat().st_size / 1024 if p.exists() else 0,
                "error": f"{type(e).__name__}: {e}"[:120]}

ada_df = paths_df[paths_df["ada"]]
sampel = pd.concat([g.sample(min(SAMPLE_IMAGES, len(g)), random_state=0)
                    for _, g in ada_df.groupby("source")]) if len(ada_df) else ada_df

if len(sampel) == 0:
    catat("WARN", "gambar", "tidak ada berkas gambar yang bisa dibuka di mesin ini — "
                            "pemeriksaan isi gambar dilewati")
    mutu = pd.DataFrame()
else:
    mutu = pd.DataFrame([periksa(p) for p in sampel["lokal"]], index=sampel.index)
    mutu["source"] = sampel["source"].values
    mutu["path"] = sampel["path"].values
    ring = mutu.groupby("source").agg(
        dicek=("ok", "size"), rusak=("ok", lambda s: int((~s).sum())),
        median_w=("w", "median"), median_h=("h", "median"),
        min_lebar=("w", "min"), median_kb=("kb", "median"),
    )
    display(ring)
    display(mutu["mode"].value_counts().to_frame("jumlah"))

    for s, r in ring.iterrows():
        if r["rusak"]:
            catat("FAIL", "gambar", f"{s}: {int(r['rusak'])}/{int(r['dicek'])} berkas sampel gagal dibuka")
        else:
            catat("PASS", "gambar", f"{s}: {int(r['dicek'])} berkas sampel semuanya bisa dibuka")

    kecil = mutu[mutu["ok"] & ((mutu["w"] < 200) | (mutu["h"] < 200))]
    catat("WARN" if len(kecil) else "PASS", "gambar",
          f"{len(kecil)}/{len(mutu)} gambar sampel beresolusi < 200 px pada satu sisi")

    if mutu["ok"].any():
        plt.figure(figsize=(11, 3.5))
        plt.scatter(mutu.loc[mutu["ok"], "w"], mutu.loc[mutu["ok"], "h"], s=6, alpha=0.4)
        plt.xlabel("lebar px"); plt.ylabel("tinggi px"); plt.title("dimensi gambar sampel")
        plt.show()
    if (~mutu["ok"]).any():
        print("contoh error:")
        display(mutu.loc[~mutu["ok"], ["source", "path", "error"]].head(5))

## 9. Lihat gambarnya sendiri

Statistik tidak menangkap gambar placeholder, watermark, atau foto yang tidak
berhubungan dengan judulnya. Mata masih perlu.

In [ ]:
def grid(n=8, sumber=None, seed=0):
    g = ada_df if sumber is None else ada_df[ada_df["source"] == sumber]
    if len(g) == 0:
        print(f"tidak ada gambar lokal untuk {sumber or 'semua sumber'}")
        return
    pilih = g.sample(min(n, len(g)), random_state=seed)
    baris = int(np.ceil(len(pilih) / 4))
    fig, axes = plt.subplots(baris, 4, figsize=(14, 3.4 * baris), squeeze=False)
    rata = np.ravel(axes)
    for ax, (_, r) in zip(rata, pilih.iterrows()):
        try:
            with Image.open(r["lokal"]) as im:
                ax.imshow(im.convert("RGB"))
            ax.set_title(Path(r["path"]).name[:28], fontsize=7)
        except Exception as e:
            ax.set_title(f"GAGAL: {type(e).__name__}", fontsize=7, color="red")
        ax.axis("off")
    for ax in rata[len(pilih):]:
        ax.axis("off")
    plt.suptitle(f"contoh gambar — {sumber or 'semua sumber'}")
    plt.tight_layout()
    plt.show()

for s in sorted(ada_df["source"].unique()):
    grid(8, sumber=s)

In [ ]:
# pasangan judul + gambar pertama, untuk memastikan gambarnya milik produk itu
cek = df[df["local_image_paths"].map(list_len) > 0].sample(6, random_state=1)
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, (_, r) in zip(np.ravel(axes), cek.iterrows()):
    p = remap(r["local_image_paths"][0])
    judul = (r["title"][:52] + "...") if len(r["title"]) > 52 else r["title"]
    if p is not None and p.exists():
        try:
            with Image.open(p) as im:
                ax.imshow(im.convert("RGB"))
        except Exception as e:
            ax.text(0.5, 0.5, f"gagal buka\n{type(e).__name__}", ha="center", va="center")
    else:
        ax.text(0.5, 0.5, "berkas tidak ada\ndi mesin ini", ha="center", va="center")
    harga_txt = "NA" if pd.isna(r["price"]) else f"Rp{int(r['price']):,}"
    ax.set_title(f"[{r['source']}] {judul}\n{harga_txt} · {r['kategori_umkm']}", fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 10. Vonis

Semua temuan dikumpulkan di sini. `FAIL` = harus diperbaiki sebelum dipakai,
`WARN` = sifat data yang harus ditangani saat penyaringan/split, `PASS` = aman.

In [ ]:
lap = pd.DataFrame(FINDINGS)
urutan = {"FAIL": 0, "WARN": 1, "PASS": 2}
lap = lap.sort_values("level", key=lambda s: s.map(urutan)).reset_index(drop=True)

print(lap["level"].value_counts().to_string(), "\n")
display(lap[lap["level"] != "PASS"])

siap = not (lap["level"] == "FAIL").any()
print("\nVONIS:", "LAYAK dipakai dengan penyaringan" if siap else "BELUM layak — ada FAIL di atas")

REPORT.write_text(json.dumps({
    "berkas": str(MERGED),
    "baris": int(len(df)),
    "kolom": int(df.shape[1]),
    "per_sumber": {k: int(v) for k, v in df["source"].value_counts().items()},
    "siap_pakai": bool(siap),
    "temuan": FINDINGS,
}, indent=2, ensure_ascii=False), encoding="utf-8")
print("laporan ->", REPORT)